# NB08 — Reextracción con pooling adaptativo (2×2)

**TFM · Máster Universitario en Inteligencia Artificial · VIU 2025-2026 · Víctor Rodríguez Rodríguez**

---

## Propósito
Reextraer las representaciones de AsymMirai sobre los 5.000 estudios de VinDr-Mammo, esta vez preservando información espacial mediante `AdaptiveAvgPool2d((2,2)) + AdaptiveMaxPool2d((2,2))` en lugar del GAP+GMP global aplicado en NB03.

**Inputs**: `Data/vindr-mammo/breast-level_annotations.csv`, DICOMs en `Data/vindr-mammo/images/`, `AsymMirai/snapshots/trained_asymmirai.pt`, `outputs/Features/metadata.csv` (opcional, reutilizado de NB03).

**Outputs**: `outputs/Features/X_view_22.npy` (~328 MB), `X_asym_22.npy` (~164 MB), `done_studies_22.txt`.

---

## Motivación

El backbone de AsymMirai devuelve un mapa `(512, 52, 64)` por vista. En el NB03 se reduce a 1024 dims (512 GAP + 512 GMP), descartando toda la información espacial. Sin embargo, los hallazgos BI-RADS sospechosos (calcificaciones, masas, asimetrías focales) son focales por naturaleza: ocupan unos pocos píxeles latentes de los 52 × 64 = 3328 disponibles. El promedio global los diluye y el máximo global los captura como pico, pero pierde la información sobre en qué región de la mama se concentra la señal.

El pooling adaptativo a una rejilla `(2,2)` divide el mapa latente en cuatro cuadrantes (superior-izquierdo, superior-derecho, inferior-izquierdo, inferior-derecho) y calcula el promedio y el máximo dentro de cada uno. Conserva la localización sin disparar la dimensionalidad como haría una rejilla más fina.

## Decisiones de diseño

- Rejilla (2,2) en lugar de (4,4) o (7,7): Con (4,4) se llegaría a 16 384 dims por vista frente a 4 096 con (2,2). Dado que el training a nivel mama tiene ~3.200 muestras (~395 positivos), (4,4) tendría un riesgo alto de sobreajuste.
- Avg + Max en lugar de solo Avg o solo Max: Avg captura magnitud distribuida, Max captura picos. Conservar ambos no aumenta significativamente el coste.
- Mismo preprocesado DICOM que en NB03 (MIRAI_MEAN/STD, alineación por centroide, redimensionado a 1664×2048): la única diferencia entre NB03 y NB08 es la función de pooling.
- Mismo `metadata.csv`: las etiquetas y los splits no cambian. Solo se regeneran las features.

In [1]:
import os, sys, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
import pydicom
from pathlib import Path

BASE = os.environ.get('TFM_PROJECT_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
ASYMMIRAI = os.path.join(BASE, 'AsymMirai')
DATA = os.path.join(BASE, 'Data', 'vindr-mammo')
OUTPUTS = os.path.join(BASE, 'outputs')
WEIGHTS = os.path.join(ASYMMIRAI, 'snapshots', 'trained_asymmirai.pt')
IMG_DIR = os.path.join(DATA, 'images')
BREAST_CSV = os.path.join(DATA, 'breast-level_annotations.csv')
FEATURES_DIR = os.path.join(OUTPUTS, 'Features')
Path(FEATURES_DIR).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, ASYMMIRAI)
sys.path.insert(0, os.path.join(ASYMMIRAI, 'asymmetry_model'))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Salida: {FEATURES_DIR}')

Device: cuda
Salida: c:\Users\victo\Documents\TFM\Proyecto\outputs\Features


## 1. Carga del modelo congelado y stretch params

Idéntico al NB03. El modelo y los pesos no cambian.

In [2]:
model = torch.load(WEIGHTS, map_location=DEVICE, weights_only=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False

USE_STRETCH = getattr(model, 'use_stretch', False)
cc_stretch  = model.cc_stretch_params.detach() if USE_STRETCH else None
mlo_stretch = model.mlo_stretch_params.detach() if USE_STRETCH else None
print(f'use_stretch = {USE_STRETCH}, alignment_space = {getattr(model,"alignment_space",None)}')

c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'mirai_localized_dif_head.LocalizedDifModel' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)


use_stretch = True, alignment_space = None


## 2. Funciones de preprocesado y extracción

La única diferencia respecto al NB03 es `pool_features_22`. Todo lo demás (`load_dicom`, `align_by_centroid`, `preprocess_view`, `compute_asym`, `extract_study`) es idéntico para garantizar que cualquier diferencia entre las features de NB03 y NB08 provenga únicamente del pooling.

### Nuevo pooling

```python
# NB03 (GAP+GMP global)
gap = x.mean(dim=(-2, -1)) 
gmp = x.amax(dim=(-2, -1))
# Concatenación -> 1024 dims por vista

# NB08 (AdaptiveAvg + AdaptiveMax a 2×2)
avg22 = F.adaptive_avg_pool2d(x, (2,2))
max22 = F.adaptive_max_pool2d(x, (2,2))
# Aplanamiento y concatenación -> 2048 + 2048 = 4096 dims por vista
```


In [3]:
MIRAI_MEAN = 7699.5
MIRAI_STD = 11765.06
TARGET_H, TARGET_W = 1664, 2048

def load_dicom(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    pixels = pixels * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0))
    if hasattr(ds, 'WindowCenter') and hasattr(ds, 'WindowWidth'):
        wc = float(ds.WindowCenter[0]) if hasattr(ds.WindowCenter, '__iter__') else float(ds.WindowCenter)
        ww = float(ds.WindowWidth[0])  if hasattr(ds.WindowWidth,  '__iter__') else float(ds.WindowWidth)
        pixels = np.clip(pixels, wc-ww/2, wc+ww/2)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        pixels = pixels.max() - pixels
    return pixels

def align_by_centroid(img):
    h, w = img.shape
    mask = (img > img.mean()).astype(np.uint8)
    M = cv2.moments(mask)
    if M['m00'] == 0:
        return img
    cy, cx = int(M['m01']/M['m00']), int(M['m10']/M['m00'])
    M_aff = np.float32([[1, 0, w//2-cx], [0, 1, h//2-cy]])
    return cv2.warpAffine(img, M_aff, (w, h), borderValue=0)

def preprocess_view(dcm_path):
    img = load_dicom(dcm_path)
    img = align_by_centroid(img)
    img = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
    img = (img - MIRAI_MEAN) / MIRAI_STD
    img = np.stack([img, img, img], axis=0)[None, ...]
    return torch.from_numpy(img).float()

def pool_features_22(x):
    # AdaptiveAvgPool2d((2,2)) + AdaptiveMaxPool2d((2,2)) sobre (B, C, H, W).
    avg = F.adaptive_avg_pool2d(x, (2, 2))
    mx = F.adaptive_max_pool2d(x, (2, 2)) 

    # Aplanamos manteniendo (canal, cuadrante) para que el orden sea consistente
    avg_flat = avg.reshape(avg.size(0), -1)
    mx_flat = mx.reshape(mx.size(0), -1)
    return torch.cat([avg_flat, mx_flat], dim=1)

def compute_asym(L, R, stretch_p=None):
    if stretch_p is not None:
        sp = stretch_p.view(1, -1, 1, 1).to(L.device)
        L = sp * L
        R = sp * R
    return torch.abs(L - R)

def extract_study(vistas):
    try:
        l_cc = preprocess_view(vistas[('L','CC')]).to(DEVICE)
        r_cc = preprocess_view(vistas[('R','CC')]).to(DEVICE)
        l_mlo = preprocess_view(vistas[('L','MLO')]).to(DEVICE)
        r_mlo = preprocess_view(vistas[('R','MLO')]).to(DEVICE)
    except Exception as e:
        print(f'error preprocesado {e}')
        return None

    with torch.no_grad():
        emb_l_cc = model.backbone(l_cc)
        emb_r_cc = model.backbone(r_cc)
        emb_l_mlo = model.backbone(l_mlo)
        emb_r_mlo = model.backbone(r_mlo)

        # Punto A: features de las 4 vistas con pooling (2,2)
        feat_l_cc = pool_features_22(emb_l_cc)[0]
        feat_r_cc = pool_features_22(emb_r_cc)[0]
        feat_l_mlo = pool_features_22(emb_l_mlo)[0]
        feat_r_mlo = pool_features_22(emb_r_mlo)[0]

        # Punto B: asimetría bilateral con stretch, también con pooling (2,2)
        asym_cc = compute_asym(emb_l_cc,  emb_r_cc,  cc_stretch  if USE_STRETCH else None)
        asym_mlo = compute_asym(emb_l_mlo, emb_r_mlo, mlo_stretch if USE_STRETCH else None)
        feat_asym_cc = pool_features_22(asym_cc)[0]
        feat_asym_mlo = pool_features_22(asym_mlo)[0]

    X_view = torch.stack([feat_l_cc, feat_r_cc, feat_l_mlo, feat_r_mlo], dim=0).cpu().numpy()
    X_asym = torch.stack([feat_asym_cc, feat_asym_mlo], dim=0).cpu().numpy()
    return X_view, X_asym

## 3. Sanity check sobre un estudio antes de lanzar

Cogemos un estudio cualquiera y comprobamos que los shapes son los esperados.

In [5]:
df = pd.read_csv(BREAST_CSV)
primer_sid = df['study_id'].iloc[0]
df_test = df[df.study_id == primer_sid]
vistas_test = {(r.laterality, r.view_position): os.path.join(IMG_DIR, primer_sid, f'{r.image_id}.dicom') for _, r in df_test.iterrows()}

if all(os.path.isfile(p) for p in vistas_test.values()):
    t0 = time.time()
    result = extract_study(vistas_test)
    if result is not None:
        Xv, Xa = result
        print(f'OK ({time.time()-t0:.1f}s)')
        print(f'X_view shape: {Xv.shape}')
        print(f'X_asym shape: {Xa.shape}')
        print(f'X_view rango: [{Xv.min():.4f}, {Xv.max():.4f}], media: {Xv.mean():.4f}')
        print(f'X_asym rango: [{Xa.min():.4f}, {Xa.max():.4f}], media: {Xa.mean():.4f}')
        assert Xv.shape == (4, 4096), 'Shape de X_view incorrecto'
        assert Xa.shape == (2, 4096), 'Shape de X_asym incorrecto'
        print('\nListo para extracción masiva.')
    else:
        print('Fallo en extract_study sobre el primer estudio')
else:
    print('Algunas vistas del primer estudio no existen en disco')

OK (0.7s)
X_view shape: (4, 4096)
X_asym shape: (2, 4096)
X_view rango: [0.0000, 2.7262], media: 0.9349
X_asym rango: [0.0000, 2.8481], media: 0.0050

Listo para extracción masiva.


## 4. Construir las etiquetas

Se reutiliza la lógica de agregación a nivel mama y estudio de NB03. Si el archivo `metadata.csv` producido por NB03 ya existe, esta celda simplemente lo lee.

In [6]:
META_PATH = os.path.join(FEATURES_DIR, 'metadata.csv')

if os.path.isfile(META_PATH):
    study_agg = pd.read_csv(META_PATH)
    print(f'metadata.csv ya existe ({len(study_agg)} filas), reutilizando.')

    # Reconstruir el diccionario de paths a partir del CSV original de VinDr
    study_imgs = {}
    for sid, grp in df.groupby('study_id'):
        study_imgs[sid] = {(r.laterality, r.view_position): r.image_id for _, r in grp.iterrows()}
else:
    print('metadata.csv no existe, generándolo de cero...')
    
    def parse_birads(v):
        if isinstance(v, str):
            s = v.replace('BI-RADS', '').replace('BIRADS', '').strip()
            try: return int(s)
            except: return np.nan
        return int(v) if not pd.isna(v) else np.nan
    
    def parse_density(v):
        if isinstance(v, str):
            return v.replace('DENSITY', '').strip()
        return v
    
    df['birads_int'] = df['breast_birads'].apply(parse_birads)
    df['density'] = df['breast_density'].apply(parse_density)
    
    birads_breast = df.groupby(['study_id','laterality']).agg(
        birads = ('birads_int', 'max'),
        density = ('density', lambda s: s.mode().iloc[0] if len(s.mode()) else 'C'),
    ).reset_index()
    birads_L = birads_breast[birads_breast.laterality == 'L'].set_index('study_id')
    birads_R = birads_breast[birads_breast.laterality == 'R'].set_index('study_id')
    
    study_agg = df.groupby('study_id').agg(
        max_birads = ('birads_int', 'max'),
        density = ('density', lambda s: s.mode().iloc[0] if len(s.mode()) else 'C'),
        split = ('split', 'first'),
    ).reset_index()

    study_agg['birads_L'] = study_agg['study_id'].map(birads_L['birads'])
    study_agg['birads_R'] = study_agg['study_id'].map(birads_R['birads'])
    study_agg['density_L'] = study_agg['study_id'].map(birads_L['density'])
    study_agg['density_R'] = study_agg['study_id'].map(birads_R['density'])
    study_agg['y_estudio'] = (study_agg['max_birads'] >= 4).astype(int)
    study_agg['y_L'] = (study_agg['birads_L'] >= 4).astype(int)
    study_agg['y_R'] = (study_agg['birads_R'] >= 4).astype(int)
    
    study_imgs = {}
    for sid, grp in df.groupby('study_id'):
        study_imgs[sid] = {(r.laterality, r.view_position): r.image_id for _, r in grp.iterrows()}
    
    print(f'Generadas {len(study_agg)} filas de metadata.')

print(f'\nEstudios: {len(study_agg)}')
print(f'Positivos nivel estudio: {study_agg.y_estudio.sum()} ({100*study_agg.y_estudio.mean():.2f}%)')
n_mamas = len(study_agg)*2; n_pos_mamas = study_agg.y_L.sum() + study_agg.y_R.sum()
print(f'Positivos nivel mama: {n_pos_mamas} de {n_mamas} ({100*n_pos_mamas/n_mamas:.2f}%)')

metadata.csv ya existe (4999 filas), reutilizando.

Estudios: 4999
Positivos nivel estudio: 481 (9.62%)
Positivos nivel mama: 494 de 9998 (4.94%)


## 5. Lógica de reanudación

Igual que en NB03: se lee `done_studies_22.txt` si existe y se salta lo ya procesado. Permite parar y continuar la extracción sin perder el progreso.

In [7]:
OUT_VIEW = os.path.join(FEATURES_DIR, 'X_view_22.npy')
OUT_ASYM = os.path.join(FEATURES_DIR, 'X_asym_22.npy')
OUT_DONE = os.path.join(FEATURES_DIR, 'done_studies_22.txt')

done = set()
if os.path.isfile(OUT_DONE):
    with open(OUT_DONE) as f:
        done = set(l.strip() for l in f if l.strip())
    print(f'Reanudando: ya hechos {len(done)} estudios')

pendientes = [s for s in study_agg.study_id.tolist() if s not in done]
print(f'Pendientes: {len(pendientes)}')

Reanudando: ya hechos 4999 estudios
Pendientes: 0


## 6. Bucle principal

Misma estructura que NB03: cada 100 estudios procesados se vuelca todo a disco. ETA mostrado cada 10 estudios.


In [8]:
X_view_buf, X_asym_buf = [], []
done_list = []
SAVE_EVERY = 100

# Precargar buffers si hay parciales
if len(done) > 0:
    if os.path.isfile(OUT_VIEW):
        X_view_buf = list(np.load(OUT_VIEW))
        X_asym_buf = list(np.load(OUT_ASYM))
    with open(OUT_DONE) as f:
        done_list = [l.strip() for l in f if l.strip()]
    print(f'Buffers precargados con {len(done_list)} estudios')

def flush():
    # Volcar los buffers a disco.
    np.save(OUT_VIEW, np.array(X_view_buf, dtype=np.float32))
    np.save(OUT_ASYM, np.array(X_asym_buf, dtype=np.float32))
    with open(OUT_DONE, 'w') as f:
        for sid in done_list:
            f.write(sid + '\n')

t_global = time.time()
errores = []

for i, sid in enumerate(pendientes, start=1):
    imgs = study_imgs.get(sid, {})
    required = [('L','CC'), ('R','CC'), ('L','MLO'), ('R','MLO')]
    if not all(k in imgs for k in required):
        errores.append((sid, 'faltan vistas en CSV')); continue
    vistas = {k: os.path.join(IMG_DIR, sid, f'{imgs[k]}.dicom') for k in required}
    if not all(os.path.isfile(p) for p in vistas.values()):
        errores.append((sid, 'DICOM ausente en disco')); continue

    t0 = time.time()
    result = extract_study(vistas)
    if result is None:
        errores.append((sid, 'fallo en extracción')); continue
    X_view, X_asym = result

    X_view_buf.append(X_view)
    X_asym_buf.append(X_asym)
    done_list.append(sid)

    if i % 10 == 0 or i == 1:
        elapsed = time.time() - t_global
        rate = elapsed / i
        eta = rate * (len(pendientes) - i) / 60
        print(f'[{i:5d}/{len(pendientes)}] sid={sid} {time.time()-t0:.1f}s/est  ETA {eta:.1f}min')

    if i % SAVE_EVERY == 0:
        flush()
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

flush()
print(f'\nExtracción completa. Procesados {len(done_list)} estudios. Errores: {len(errores)}')
if errores:
    print('Primeros errores:')
    for sid, err in errores[:5]:
        print(f'{sid} {err}')

Buffers precargados con 4999 estudios

Extracción completa. Procesados 4999 estudios. Errores: 0


## 7. Verificación final

Comprobar shapes y consistencia con el metadata.csv (las filas deben estar en el mismo orden que en NB03).

In [9]:
X_view_22 = np.load(OUT_VIEW)
X_asym_22 = np.load(OUT_ASYM)

print(f'X_view_22: shape={X_view_22.shape} {X_view_22.nbytes/1e6:.1f} MB')
print(f'X_asym_22: shape={X_asym_22.shape} {X_asym_22.nbytes/1e6:.1f} MB')
print(f'\nMetadata filas: {len(study_agg)}')

assert X_view_22.shape[0] == len(study_agg), 'Desalineación entre X_view_22 y metadata'
assert X_asym_22.shape[0] == len(study_agg), 'Desalineación entre X_asym_22 y metadata'
assert X_view_22.shape[1:] == (4, 4096), f'Shape de X_view_22 inesperado: {X_view_22.shape}'
assert X_asym_22.shape[1:] == (2, 4096), f'Shape de X_asym_22 inesperado: {X_asym_22.shape}'

# Estadísticas comparativas
X_view_orig = np.load(os.path.join(FEATURES_DIR, 'X_view.npy'))
X_asym_orig = np.load(os.path.join(FEATURES_DIR, 'X_asym.npy'))

print(f'\nComparación de magnitudes (GAP+GMP vs pooling 2,2)')
print(f'X_view original (GAP+GMP, 1024 dims): media={X_view_orig.mean():.4f}, std={X_view_orig.std():.4f}, max={X_view_orig.max():.4f}')
print(f'X_view nuevo (pool2x2, 4096 dims): media={X_view_22.mean():.4f}, std={X_view_22.std():.4f}, max={X_view_22.max():.4f}')
print(f'X_asym original (GAP+GMP, 1024 dims): media={X_asym_orig.mean():.4f}, std={X_asym_orig.std():.4f}, max={X_asym_orig.max():.4f}')
print(f'X_asym nuevo (pool2x2, 4096 dims): media={X_asym_22.mean():.4f}, std={X_asym_22.std():.4f}, max={X_asym_22.max():.4f}')

X_view_22: shape=(4999, 4, 4096) 327.6 MB
X_asym_22: shape=(4999, 2, 4096) 163.8 MB

Metadata filas: 4999

Comparación de magnitudes (GAP+GMP vs pooling 2,2)
X_view original (GAP+GMP, 1024 dims): media=0.9661, std=0.4652, max=8.8575
X_view nuevo (pool2x2, 4096 dims): media=0.9395, std=0.4531, max=8.8575
X_asym original (GAP+GMP, 1024 dims): media=0.0128, std=0.1245, max=10.1893
X_asym nuevo (pool2x2, 4096 dims): media=0.0090, std=0.0940, max=10.1893
